# SolverNode 테스트 노트북

SolverNode의 TTA(Test Time Augmentation) 기반 추론을 테스트합니다.

## 테스트 항목
- 선지 셔플링 및 리맵핑
- Track A (비문학) / Track B (RAG) 분기
- 실제 데이터 일부에 대한 추론

In [ ]:
# [Cell 1] 환경 설정
import sys
import os
import pandas as pd

# 프로젝트 루트 경로 추가
current_dir = os.getcwd()
project_root = os.path.dirname(current_dir)
if project_root not in sys.path:
    sys.path.append(project_root)

# config 로드
from src.utils.config_loader import load_config

cfg = load_config()

print("✅ 설정 로드 완료!")
print(f"🔹 메인 모델: {cfg.model.main_solver.path}")
print(f"🔹 데이터 경로: {cfg.path.data.validate}")

In [ ]:
# [Cell 2] 데이터 로드
import json

# 실제 데이터 로드 시도
data_path = os.path.join(project_root, cfg.path.data.validate)

if os.path.exists(data_path):
    df = pd.read_csv(data_path)
    print(f"✅ 데이터 로드 완료: {len(df)}개 샘플")
    print(f"🔹 컬럼: {list(df.columns)}")
    display(df.head(3))
else:
    print(f"⚠️ 데이터 파일이 없습니다: {data_path}")
    print("📝 샘플 데이터를 사용합니다.")
    
    # 샘플 데이터 생성
    sample_data = [
        {
            "id": "sample_001",
            "paragraph": """디지털 시대에 프라이버시의 개념은 크게 변화하고 있다. 
과거에는 물리적 공간의 침해가 프라이버시 침해의 주된 형태였다면, 
오늘날에는 개인 정보의 수집과 활용이 핵심 쟁점이 되었다. 
특히 빅데이터와 인공지능 기술의 발전으로 개인의 행동 패턴을 예측하고 
분석하는 것이 가능해지면서, 프라이버시 보호의 범위와 방법에 대한 
새로운 논의가 필요해졌다.""",
            "question": "윗글의 내용과 일치하는 것은?",
            "choices": [
                "과거의 프라이버시 침해는 주로 정보 수집과 관련되었다.",
                "빅데이터 기술은 프라이버시 보호를 강화하는 데 기여한다.",
                "디지털 시대에는 개인 정보 활용이 프라이버시의 핵심 쟁점이다.",
                "인공지능은 개인 행동 예측과 무관하다.",
                "프라이버시 보호 방법은 변화할 필요가 없다."
            ],
            "answer": 3
        },
        {
            "id": "sample_002",
            "paragraph": """조선 시대의 과거 제도는 관료 선발의 핵심 기제였다. 
과거는 크게 문과, 무과, 잡과로 나뉘었으며, 이 중 문과가 가장 중요시되었다. 
문과에 급제하면 관직에 진출할 수 있었고, 이는 양반 신분을 유지하는 
핵심적인 방법이었다. 그러나 과거 응시 자격은 양인 이상에게만 주어졌으며, 
천민은 응시할 수 없었다.""",
            "question": "조선 시대 과거 제도에 대한 설명으로 옳은 것은?",
            "choices": [
                "무과가 문과보다 중요하게 여겨졌다.",
                "천민도 과거에 응시할 수 있었다.",
                "문과 급제는 관직 진출의 기회를 제공했다.",
                "과거는 문과와 무과 두 종류만 있었다.",
                "양반 신분과 과거는 무관했다."
            ],
            "answer": 3
        }
    ]
    
    df = pd.DataFrame(sample_data)
    print(f"✅ 샘플 데이터 생성: {len(df)}개")

In [ ]:
# [Cell 3] SolverNode 로드
from src.agent.nodes.solver import SolverNode

print("⏳ SolverNode 초기화 중... (모델 로딩에 시간이 걸릴 수 있습니다)")
solver = SolverNode(cfg)
print("✅ SolverNode 초기화 완료!")

In [ ]:
# [Cell 4] 단일 문제 테스트

# 첫 번째 샘플 선택
sample = df.iloc[0]

# State 구성
state = {
    "paragraph": sample["paragraph"],
    "problem": {
        "question": sample["question"],
        "choices": sample["choices"] if isinstance(sample["choices"], list) else json.loads(sample["choices"])
    },
    "track_info": {
        "is_rag_required": "false",
        "category": "비문학"
    }
}

print("📝 테스트 문제:")
print(f"   지문: {state['paragraph'][:100]}...")
print(f"   질문: {state['problem']['question']}")
print(f"   정답: {sample.get('answer', 'N/A')}")
print("\n" + "="*50)

# SolverNode 실행
result = solver(state)

print("\n📊 결과:")
for res in result["solver_results"]:
    print(f"   [V{res['version']}] 셔플순서: {res['shuffled_order']} → 셔플답: {res['shuffled_answer']} → 원본답: {res['answer']}")

In [ ]:
# [Cell 5] TTA 결과 분석
from collections import Counter

# 답안 분포 확인
answers = [res["answer"] for res in result["solver_results"]]
answer_counts = Counter(answers)

print("📊 TTA 답안 분포:")
for ans, count in sorted(answer_counts.items()):
    bar = "█" * count
    print(f"   {ans}번: {bar} ({count}표)")

# 다수결 결과
majority_answer = answer_counts.most_common(1)[0][0]
print(f"\n🎯 다수결 결과: {majority_answer}번")

# 정답 비교 (정답이 있는 경우)
if "answer" in sample:
    correct = sample["answer"]
    is_correct = majority_answer == correct
    print(f"✅ 정답: {correct}번 → {'정답!' if is_correct else '오답'}") 

In [ ]:
# [Cell 6] 여러 문제 배치 테스트
from collections import Counter

def run_solver_and_vote(solver, sample):
    """SolverNode 실행 후 다수결 투표"""
    state = {
        "paragraph": sample["paragraph"],
        "problem": {
            "question": sample["question"],
            "choices": sample["choices"] if isinstance(sample["choices"], list) else json.loads(sample["choices"])
        },
        "track_info": {
            "is_rag_required": "false",
            "category": "비문학"
        }
    }
    
    result = solver(state)
    answers = [res["answer"] for res in result["solver_results"]]
    majority = Counter(answers).most_common(1)[0][0]
    
    return majority, result

# 테스트할 샘플 수 (데이터 크기에 맞게 조절)
num_samples = min(3, len(df))
correct_count = 0

print(f"🚀 {num_samples}개 문제 테스트 시작\n")

for idx in range(num_samples):
    sample = df.iloc[idx]
    predicted, _ = run_solver_and_vote(solver, sample)
    
    if "answer" in sample:
        correct = sample["answer"]
        is_correct = predicted == correct
        if is_correct:
            correct_count += 1
        status = "✅" if is_correct else "❌"
        print(f"[{idx+1}/{num_samples}] {status} 예측: {predicted}번, 정답: {correct}번")
    else:
        print(f"[{idx+1}/{num_samples}] 예측: {predicted}번")

if "answer" in df.columns:
    accuracy = correct_count / num_samples * 100
    print(f"\n📈 정확도: {correct_count}/{num_samples} ({accuracy:.1f}%)")

In [ ]:
# [Cell 7] 내부 함수 단위 테스트 (모델 로딩 없이)

print("🧪 SolverNode 내부 함수 테스트\n")

# 1. _format_choices 테스트
print("1️⃣ _format_choices 테스트:")
choices = ["선지A", "선지B", "선지C"]
formatted = solver._format_choices(choices)
print(f"   입력: {choices}")
print(f"   출력:\n{formatted}")

# 2. _parse_answer 테스트
print("\n2️⃣ _parse_answer 테스트:")
test_outputs = [
    "정답: 3",
    "정답은 ②입니다",
    "<think>분석...</think>정답: 1",
    "4번이 정답입니다"
]
for output in test_outputs:
    parsed = solver._parse_answer(output)
    print(f"   '{output[:30]}...' → {parsed}번")

# 3. _remap_index 테스트
print("\n3️⃣ _remap_index 테스트:")
shuffled_order = [3, 1, 4, 2, 5]  # 셔플된 1번 위치에 원본 3번이 있음
print(f"   셔플 순서: {shuffled_order}")
for i in range(1, 6):
    original = solver._remap_index(i, shuffled_order)
    print(f"   셔플 {i}번 → 원본 {original}번")

# 4. _generate_tta_versions 테스트
print("\n4️⃣ _generate_tta_versions 테스트:")
choices = ["A", "B", "C", "D", "E"]
versions = solver._generate_tta_versions(choices)
print(f"   원본 선지: {choices}")
print(f"   생성된 TTA 버전: {len(versions)}개")
for i, (shuffled, indices) in enumerate(versions):
    print(f"   V{i+1}: {shuffled} (순서: {indices})")